# Deep Learning in Computational Mechanics: An Introductory Course (2nd Ed.) -- Capitulo 4, PINN para la barra estatica

**Libro:** Herrmann, L., Jokeit, M., Weeger, O., Kollmannsberger, S. (2025). *Deep Learning in Computational Mechanics: An Introductory Course* (2nd ed.). Springer.

**Carpeta origen:** `PINNs/4. Otros/Deep Learning in Computational Mechanics.pdf`

## Como se usan las PINNs en este libro

El Capitulo 4 ("Introduction to Physics-Informed Neural Networks") introduce las PINNs con el **ejemplo mas simple posible**: una barra elastica lineal 1D en equilibrio estatico (Fig. 4.2), gobernada por (Eq. 4.4-4.6):

$$\frac{d}{dx}\Big(EA\frac{du}{dx}\Big)+p=0 \text{ en } \Omega,\qquad EA\frac{du}{dx}=F \text{ en } \Gamma_N,\qquad u=g \text{ en } \Gamma_D$$

El caso concreto que desarrolla el libro (Seccion 4.2.1) usa $\Omega=[0,1]$, $EA=1$, y el **metodo de la solucion manufacturada**: se elige $u(x)=\sin(2\pi x)$ y se sustituye en la EDO para obtener la carga distribuida correspondiente $p(x)=4\pi^2\sin(2\pi x)$ (Eq. 4.8-4.9), con condiciones de contorno de Dirichlet homogeneas $u(0)=u(1)=0$.

La red (Fig. 4.3) es una MLP simple de una sola capa oculta con activacion tanh, que predice $\hat u(x)$; el residuo $r(x)=\frac{d}{dx}(EA\frac{du}{dx})+p$ se calcula por diferenciacion automatica (Listado 4.1). La funcion de costo combina la perdida de frontera $\mathcal{L}_B$ y la perdida del residuo $\mathcal{L}_R$ (Eq. 4.11-4.13), entrenada con L-BFGS (Algoritmo 9).

Este cuaderno reproduce **literalmente** el ejemplo del libro, incluyendo el codigo Python casi identico al que aparece impreso en el texto (Listado 4.1 y los fragmentos de las paginas 130-131), incluyendo los mismos hiperparametros (10 puntos de colocacion, una capa oculta, entrenamiento con L-BFGS).

## Repositorio publico

El propio libro declara explicitamente su repositorio de codigo en el Prefacio: "the accompanying files are available at **www.deeplearningincomputationalmechanics.com**", con implementaciones en PyTorch bajo licencia CC BY-NC-SA 4.0. Ese sitio es el repositorio oficial de ejercicios y ejemplos (incluyendo el Ejercicio E.14 "Physics-Informed Neural Network for a Static Bar", que es exactamente el ejemplo reproducido aqui).

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Red del libro (pag. 130): una capa oculta, activacion tanh

In [ ]:
inputDim, hiddenDim, outputDim = 1, 20, 1

model = torch.nn.Sequential(torch.nn.Linear(inputDim, hiddenDim),
                             torch.nn.Tanh(),
                             torch.nn.Linear(hiddenDim, outputDim)).to(device)

uPred = model(torch.tensor([[0.5]], device=device))
print('Prediccion inicial (sin entrenar) en x=0.5:', uPred.item())

## 2. Listado 4.1: diferenciacion automatica envuelta, y el residuo $r(x)$ (Eq. 4.7)

In [ ]:
def getDerivative(y, x):
    dydx = torch.autograd.grad(y, x, torch.ones_like(y),
                                create_graph=True, retain_graph=True)[0]
    return dydx


def r(model, x, EA, p):
    u = model(x)
    dudx = getDerivative(u, x)
    dEAdudxx = getDerivative(EA(x) * dudx, x)
    r = dEAdudxx + p(x)
    return r

## 3. Ejemplo concreto (Seccion 4.2.1): $EA=1$, $u(x)=\sin(2\pi x) \Rightarrow p(x)=4\pi^2\sin(2\pi x)$, $u(0)=u(1)=0$

In [ ]:
x = torch.linspace(0, 1, 10, requires_grad=True, device=device).unsqueeze(1)
EA = lambda x: 1 + 0 * x
p = lambda x: 4 * torch.pi ** 2 * torch.sin(2 * torch.pi * x)

def exact_u(x):
    return np.sin(2 * np.pi * x)

u0 = 0
u1 = 0

## 4. Funcion de costo (Eq. 4.11-4.13) y entrenamiento con L-BFGS (Algoritmo 9)

In [ ]:
history = {'C': [], 'LR': [], 'LB': []}

def closure():
    optimizer.zero_grad()
    rPred = r(model, x, EA, p)
    lossR = torch.sum(rPred ** 2)

    u0Pred = model(torch.tensor([[0.]], device=device))
    u1Pred = model(torch.tensor([[1.]], device=device))
    lossB = (u0Pred - u0) ** 2 + (u1Pred - u1) ** 2

    cost = lossR + lossB.squeeze()
    cost.backward()
    history['C'].append(cost.item())
    history['LR'].append(lossR.item())
    history['LB'].append(lossB.item())
    return cost


optimizer = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=150,
                               history_size=50, line_search_fn='strong_wolfe')
optimizer.step(closure)
print(f'Costo final: {history["C"][-1]:.6e}  (en {len(history["C"])} evaluaciones de closure)')

## 5. Resultados (cf. Fig. 4.4 del libro): desplazamientos y evolucion del costo

In [ ]:
x_plot = np.linspace(0, 1, 200)
with torch.no_grad():
    u_pred = model(torch.tensor(x_plot, dtype=torch.float32, device=device).unsqueeze(1)).cpu().numpy().flatten()
u_ex = exact_u(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(x_plot, u_ex, 'o', markersize=3, color='gray', label='analytical solution')
axes[0].plot(x_plot, u_pred, 'k--', label='prediction')
axes[0].set_xlabel('x'); axes[0].set_ylabel('displacement u')
axes[0].legend(); axes[0].set_title('(a) Displacements')

axes[1].semilogy(history['C'], 'k-', label='cost C')
axes[1].semilogy(history['LR'], 'r:', label='$L_R$')
axes[1].semilogy(history['LB'], 'gray', linestyle='--', label='$L_B$')
axes[1].set_xlabel('evaluaciones'); axes[1].set_ylabel('cost')
axes[1].legend(); axes[1].set_title('(b) Cost function history')
plt.tight_layout()
plt.show()

err = 100 * np.linalg.norm(u_pred - u_ex) / np.linalg.norm(u_ex)
print(f'Error relativo L2: {err:.4f}%')